# Entrenar TODOS los modelos secuencialmente (versión optimizada)

Versión optimizada con:
- **Rank LoRA inverso al tamaño**: `r=16` para 3B/4B (LoRA puro), `r=64` para 7B (QLoRA).
- **Alpha proporcional**: factor 2.0 cuando `r` es bajo (`alpha = 2*r`).
- **Early stopping patience=1**: para si `eval_loss` no mejora en 1 epoch.
- **`num_train_epochs=5`** como tope, pero early stopping cortará antes.

Cada modelo se entrena, evalúa y guarda en `../results/<model_short>.json`. La función libera memoria GPU entre modelos. Si uno falla, continúa con el siguiente.

**Lanza todo y vete.**

## 1. Setup global

In [ ]:
import os
import gc
import json
import time
from pathlib import Path
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv()

import comet_ml
import torch

assert os.environ.get("HF_TOKEN", "").startswith("hf_"), "Falta HF_TOKEN en .env"
assert os.environ.get("COMET_API_KEY"), "Falta COMET_API_KEY en .env"
print("Tokens cargados.")

## 2. Descargar dataset (una vez para todos)

In [ ]:
from huggingface_hub import snapshot_download

HF_DATASET = "damianGil/wildfire-prevention"

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data" / "wildfire"
RESULTS_DIR  = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=HF_DATASET,
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print(f"Dataset listo en {DATA_DIR}")

## 3. Constantes compartidas

In [ ]:
SYSTEM_PROMPT = """\
You are a remote sensing analyst specialising in wildfire risk assessment.
You will be given two Sentinel-2 satellite images of the same land tile:
  1. RGB composite (bands B4-B3-B2): natural colour, useful for terrain, infrastructure, and land cover.
  2. SWIR composite (bands B12-B8-B4): highlights vegetation moisture stress and dryness. Healthy vegetation appears green/cyan, stressed or dry vegetation appears orange/red, bare soil appears magenta/pink, and burned areas appear dark red or black.

Assess the wildfire risk of the tile and return ONLY a valid JSON object — no markdown, no explanation outside the JSON — with exactly these fields:

{
  "risk_level": "low | medium | high",
  "dry_vegetation_present": true | false,
  "urban_interface": true | false,
  "steep_terrain": true | false,
  "water_body_present": true | false,
  "image_quality_limited": true | false
}
"""

USER_TEXT = (
    "Image 1 is the RGB composite. Image 2 is the SWIR composite. "
    "Return the wildfire risk JSON for this tile."
)
PROMPT = f"{SYSTEM_PROMPT.strip()}\n\n{USER_TEXT}"

FIELDS = ["risk_level", "dry_vegetation_present", "urban_interface",
          "steep_terrain", "water_body_present", "image_quality_limited"]

## 4. Función `train_one_model` (con early stopping)

In [ ]:
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from datasets import load_dataset
from PIL import Image


def to_unsloth_conversation(sample, data_dir):
    rgb  = Image.open(data_dir / sample["rgb_path"]).convert("RGB")
    swir = Image.open(data_dir / sample["swir_path"]).convert("RGB")
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": rgb},
                {"type": "image", "image": swir},
                {"type": "text",  "text":  PROMPT},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["output"]},
            ]},
        ]
    }


def free_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def train_one_model(
    model_name: str,
    model_short: str,
    load_in_4bit: bool = False,
    per_device_train_batch_size: int = 4,
    gradient_accumulation_steps: int = 2,
    num_train_epochs_max: int = 5,
    early_stopping_patience: int = 1,
    learning_rate: float = 1e-4,
    r: int = 16,
    lora_alpha: int = 32,
) -> dict:
    """Entrena un VLM con LoRA + early stopping. Guarda JSON en RESULTS_DIR."""
    print(f"\n{'='*70}\n  ENTRENANDO {model_short}  ({model_name})\n  r={r}, alpha={lora_alpha}, 4bit={load_in_4bit}\n{'='*70}\n")
    
    free_gpu_memory()
    
    print("[1/6] Cargando modelo base...")
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name     = model_name,
        max_seq_length = 4096,
        load_in_4bit   = load_in_4bit,
    )
    
    print("[2/6] Configurando LoRA...")
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = True,
        finetune_language_layers   = True,
        finetune_attention_modules = True,
        finetune_mlp_modules       = True,
        r            = r,
        lora_alpha   = lora_alpha,
        lora_dropout = 0.05,
        bias         = "none",
        random_state = 42,
    )
    
    print("[3/6] Preparando dataset...")
    ds = load_dataset(str(DATA_DIR))
    train_dataset = [to_unsloth_conversation(s, DATA_DIR) for s in ds["train"]]
    test_dataset  = [to_unsloth_conversation(s, DATA_DIR) for s in ds["test"]]
    
    print("[4/6] Entrenando (con early stopping)...")
    FastVisionModel.for_training(model)
    
    cfg_label = "qlora" if load_in_4bit else "lora"
    
    trainer = SFTTrainer(
        model         = model,
        tokenizer     = tokenizer,
        data_collator = UnslothVisionDataCollator(model, tokenizer),
        train_dataset = train_dataset,
        eval_dataset  = test_dataset,
        callbacks     = [EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
        args = SFTConfig(
            per_device_train_batch_size = per_device_train_batch_size,
            gradient_accumulation_steps = gradient_accumulation_steps,
            num_train_epochs            = num_train_epochs_max,
            warmup_ratio                = 0.03,
            learning_rate               = learning_rate,
            lr_scheduler_type           = "cosine",
            logging_steps               = 5,
            optim                       = "adamw_8bit",
            weight_decay                = 0.001,
            seed                        = 42,
            output_dir                  = f"outputs/{model_short}",
            run_name                    = f"{model_short}-{cfg_label}-r{r}-alpha{lora_alpha}-vision-on-early",
            report_to                   = ["comet_ml"],
            eval_strategy               = "epoch",
            per_device_eval_batch_size  = per_device_train_batch_size,
            save_strategy               = "epoch",
            save_total_limit            = 2,
            load_best_model_at_end      = True,
            metric_for_best_model       = "eval_loss",
            greater_is_better           = False,
            remove_unused_columns       = False,
            dataset_text_field          = "",
            dataset_kwargs              = {"skip_prepare_dataset": True},
            max_length                  = 4096,
        ),
    )
    
    torch.cuda.reset_peak_memory_stats()
    trainer_stats = trainer.train()
    peak_mem = torch.cuda.max_memory_reserved() / 1024**3
    actual_epochs = trainer_stats.metrics.get('epoch', 0)
    
    print("[5/6] Evaluando sobre todo el test set...")
    FastVisionModel.for_inference(model)
    
    correct = defaultdict(int)
    valid_json_count = 0
    inf_times = []
    N_EVAL = len(ds["test"])
    
    for i in range(N_EVAL):
        s = ds["test"][i]
        rgb  = Image.open(DATA_DIR / s["rgb_path"]).convert("RGB")
        swir = Image.open(DATA_DIR / s["swir_path"]).convert("RGB")
        gt   = json.loads(s["output"])
        
        msgs = [{"role": "user", "content": [
            {"type": "image"}, {"type": "image"}, {"type": "text", "text": PROMPT}
        ]}]
        text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
        inp  = tokenizer([rgb, swir], text, add_special_tokens=False, return_tensors="pt").to("cuda")
        
        t0 = time.time()
        out_ids = model.generate(**inp, max_new_tokens=256, use_cache=True,
                                  temperature=0.1, min_p=0.1)
        inf_times.append(time.time() - t0)
        
        pred_text = tokenizer.decode(out_ids[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        
        try:
            clean = pred_text.strip()
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            pred = json.loads(clean)
            valid_json_count += 1
            for f in FIELDS:
                if pred.get(f) == gt.get(f):
                    correct[f] += 1
        except (json.JSONDecodeError, IndexError):
            pass
        
        if (i + 1) % 50 == 0:
            print(f"      {i+1}/{N_EVAL} muestras evaluadas")
    
    valid_json_rate    = valid_json_count / N_EVAL
    accuracy_per_field = {f: correct[f] / N_EVAL for f in FIELDS}
    overall            = sum(correct.values()) / (N_EVAL * len(FIELDS))
    avg_inf_time       = sum(inf_times) / len(inf_times)
    
    print("[6/6] Guardando resultados...")
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params     = sum(p.numel() for p in model.parameters())
    
    results = {
        "model_name":  model_name,
        "model_short": model_short,
        "training": {
            "num_params_total":     total_params,
            "num_params_trainable": trainable_params,
            "vram_peak_gb":         peak_mem,
            "training_time_min":    trainer_stats.metrics['train_runtime'] / 60,
            "epochs_actual":        actual_epochs,
            "loss_train_final":     trainer_stats.training_loss,
            "config": {
                "r": r, "lora_alpha": lora_alpha, "lora_dropout": 0.05,
                "finetune_vision_layers":  True,
                "load_in_4bit":            load_in_4bit,
                "num_train_epochs_max":    num_train_epochs_max,
                "early_stopping_patience": early_stopping_patience,
                "batch_effective":         per_device_train_batch_size * gradient_accumulation_steps,
                "learning_rate":           learning_rate,
                "lr_scheduler":            "cosine",
                "warmup_ratio":            0.03,
            },
        },
        "eval": {
            "num_samples":          N_EVAL,
            "valid_json":           valid_json_rate,
            "accuracy_per_field":   accuracy_per_field,
            "overall":              overall,
            "inference_time_per_sample_s": avg_inf_time,
        },
    }
    out_path = RESULTS_DIR / f"{model_short}.json"
    out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(f"\n>>> {model_short}: overall={overall:.3f}, vram_peak={peak_mem:.2f}GB, train={trainer_stats.metrics['train_runtime']/60:.1f}min, epochs={actual_epochs:.1f}")
    print(f"    Guardado en: {out_path}")
    
    del trainer, model, tokenizer, train_dataset, test_dataset, ds
    free_gpu_memory()
    
    return results

## 5. Lanzar bucle: los 3 modelos secuencialmente

In [ ]:
MODELS_TO_TRAIN = [
    {
        "model_name":  "unsloth/Qwen2.5-VL-3B-Instruct",
        "model_short": "qwen25_vl_3b_r16",
        "load_in_4bit": False,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 2,
        "r": 16, "lora_alpha": 32,
    },
    {
        "model_name":  "unsloth/Qwen2.5-VL-7B-Instruct",
        "model_short": "qwen25_vl_7b",
        "load_in_4bit": True,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "r": 64, "lora_alpha": 64,
    },
    {
        "model_name":  "unsloth/gemma-3-4b-it",
        "model_short": "gemma3_4b",
        "load_in_4bit": False,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 2,
        "r": 16, "lora_alpha": 32,
    },
]

all_results = []
for cfg in MODELS_TO_TRAIN:
    try:
        result = train_one_model(**cfg)
        all_results.append(result)
    except Exception as e:
        print(f"\n!!! FALLÓ {cfg['model_short']}: {type(e).__name__}: {e}")
        print("    Continuando con el siguiente...\n")
        free_gpu_memory()
        continue

print(f"\n{'='*70}\n  TERMINADO. {len(all_results)}/{len(MODELS_TO_TRAIN)} modelos OK.\n{'='*70}")
for r in all_results:
    print(f"  {r['model_short']:25s} overall={r['eval']['overall']:.3f}  train={r['training']['training_time_min']:.1f}min  epochs={r['training']['epochs_actual']:.1f}")